# 📊 Análise Exploratória de Dados (EDA) - Previsão de Churn

Este notebook realiza uma análise completa do dataset Telco Customer Churn, identificando padrões, distribuições e relacionamentos entre variáveis.

## 1. Carregamento e Exploração Inicial dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo dos gráficos
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Carregar dados
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Exibir informações básicas
print(f"Shape do dataset: {df.shape}")
print(f"\nPrimeiras linhas:")
print(df.head())
print(f"\nTipos de dados:")
print(df.dtypes)
print(f"\nInformações gerais:")
df.info()

## 2. Análise de Valores Nulos

In [ ]:
# Verificar valores nulos
missing_values = pd.DataFrame({
    'Coluna': df.columns,
    'Nulos': df.isnull().sum(),
    'Percentual (%)': (df.isnull().sum() / len(df) * 100).round(2)
})

print("Análise de Valores Nulos:")
print(missing_values[missing_values['Nulos'] > 0])

# Heatmap de valores nulos
if df.isnull().sum().sum() > 0:
    plt.figure(figsize=(12, 6))
    sns.heatmap(df.isnull(), cbar=True, cmap='viridis', yticklabels=False)
    plt.title('Mapa de Valores Nulos')
    plt.tight_layout()
    plt.savefig('../reports/figures/01_missing_values.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("✓ Não há valores nulos no dataset")

## 3. Análise Univariada — Variáveis Numéricas

In [ ]:
# Identificar variáveis numéricas
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Variáveis numéricas: {numeric_cols}")

# Estatísticas descritivas
print("\nEstatísticas Descritivas:")
print(df[numeric_cols].describe())

# Histogramas
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(15, 4))
for idx, col in enumerate(numeric_cols):
    axes[idx].hist(df[col], bins=30, color='skyblue', edgecolor='black')
    axes[idx].set_title(f'Distribuição de {col}')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequência')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/02_numeric_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Análise Univariada — Variáveis Categóricas

In [ ]:
# Identificar variáveis categóricas
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Variáveis categóricas: {categorical_cols}")

# Principais categóricas para análise
main_categorical = ['Contract', 'PaymentMethod', 'InternetService', 'gender']

# Gráficos de distribuição categórica
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for idx, col in enumerate(main_categorical):
    if col in df.columns:
        value_counts = df[col].value_counts()
        axes[idx].bar(range(len(value_counts)), value_counts.values, color='coral')
        axes[idx].set_xticks(range(len(value_counts)))
        axes[idx].set_xticklabels(value_counts.index, rotation=45, ha='right')
        axes[idx].set_title(f'Distribuição de {col}')
        axes[idx].set_ylabel('Frequência')
        
        # Adicionar percentual
        percentages = (value_counts.values / len(df) * 100).round(1)
        for i, (v, p) in enumerate(zip(value_counts.values, percentages)):
            axes[idx].text(i, v, f'{p}%', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('../reports/figures/03_categorical_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Análise do Target — Distribuição de Churn

In [ ]:
# Análise do target
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

print("Distribuição do Churn:")
print(churn_counts)
print(f"\nPercentual:")
print(churn_pct)
print(f"\n⚠️ Desbalanceamento: {churn_pct.min():.2f}% vs {churn_pct.max():.2f}%")

# Gráficos
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(churn_counts.values, labels=churn_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[0].set_title('Proporção de Churn (Pie Chart)')

# Bar plot
axes[1].bar(churn_counts.index, churn_counts.values, color=colors)
axes[1].set_title('Contagem de Churn (Bar Plot)')
axes[1].set_ylabel('Número de Clientes')
axes[1].set_xlabel('Churn')

for i, v in enumerate(churn_counts.values):
    axes[1].text(i, v, f'{v}\n({churn_pct.iloc[i]:.1f}%)', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('../reports/figures/04_churn_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Análise Bivariada — Relação com Churn

In [ ]:
# Boxplots: variáveis numéricas vs Churn
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(15, 4))

for idx, col in enumerate(numeric_cols):
    sns.boxplot(data=df, x='Churn', y=col, ax=axes[idx], palette=['#2ecc71', '#e74c3c'])
    axes[idx].set_title(f'{col} vs Churn')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/05_numeric_churn_relationship.png', dpi=300, bbox_inches='tight')
plt.show()

# Taxa de churn por categoria
print("Taxa de Churn por Categoria:\n")
for col in main_categorical:
    if col in df.columns:
        churn_by_cat = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').sum() / len(x) * 100)
        print(f"{col}:")
        print(churn_by_cat.round(2))
        print()

## 7. Gráficos de Taxa de Churn por Variável Categórica

In [ ]:
# Taxa de churn para principais variáveis
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for idx, col in enumerate(main_categorical):
    if col in df.columns:
        churn_data = df.groupby(col).agg({
            'Churn': lambda x: (x == 'Yes').sum()
        }).reset_index()
        
        total_data = df.groupby(col).size().reset_index(name='Total')
        churn_data = churn_data.merge(total_data, on=col)
        churn_data['churn_rate'] = (churn_data['Churn'] / churn_data['Total'] * 100).round(2)
        
        axes[idx].barh(churn_data[col].astype(str), churn_data['churn_rate'], color='#e74c3c')
        axes[idx].set_title(f'Taxa de Churn por {col} (%)')
        axes[idx].set_xlabel('Taxa de Churn (%)')
        
        for i, v in enumerate(churn_data['churn_rate']):
            axes[idx].text(v, i, f' {v}%', va='center')

plt.tight_layout()
plt.savefig('../reports/figures/06_churn_rate_by_category.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Análise de Correlação

In [ ]:
# Converter Churn para numérico para correlação
df_corr = df.copy()
df_corr['Churn_numeric'] = (df_corr['Churn'] == 'Yes').astype(int)

# Matriz de correlação das variáveis numéricas
correlation_matrix = df_corr[numeric_cols + ['Churn_numeric']].corr()

print("Correlação com Churn:")
churn_corr = correlation_matrix['Churn_numeric'].sort_values(ascending=False)
print(churn_corr)

# Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlação - Variáveis Numéricas')
plt.tight_layout()
plt.savefig('../reports/figures/07_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Resumo e Insights Principais

### 💡 Top 5 Insights encontrados:

1. **Desbalanceamento de Classes**: O dataset possui cerca de 26% de churn e 74% sem churn, indicando desbalanceamento que será tratado durante o treinamento com `class_weight='balanced'` ou SMOTE.

2. **Tenure e TotalCharges são fortemente correlacionados**: Clientes com menor tempo de contrato (tenure) têm maior probabilidade de churn, sugerindo que manter clientes nos primeiros meses é crítico.

3. **Tipo de Contrato é um forte preditor**: Clientes com contrato mensal (Month-to-month) têm taxa de churn significativamente maior (~42%) comparado a contratos anuais (~11%) e de 2 anos (~3%).

4. **Serviços de Retenção são protetivos**: Clientes com serviços como Technical Support, Online Security, e Device Protection têm menor taxa de churn, indicando que up-selling desses serviços pode reduzir churn.

5. **MonthlyCharges elevados aumentam churn**: Clientes pagando mensalidades mais altas têm tendência a maior churn, especialmente quando combinado com contrato mensal. Estratégia de preço pode ser relevante.

### 📌 Próximas Etapas:
- Pré-processar dados (tratamento de categorias, normalização)
- Treinar múltiplos modelos de classificação
- Avaliar performance e otimizar hiperparâmetros
- Deployar modelo em API